# Semantic Pipeline Follow-up

This notebook extends the post-pull support analysis with:
- 16-image query-only semantic results on the support set
- direct comparison against the detector-only low-threshold baseline
- controller prompt-variant comparisons for the main failure cases
- a record of the long unattended experiment queue

In [1]:
from pathlib import Path
import json
import pandas as pd

REPO = Path('/home/david/semantic_vlm_privacy')
OUT = REPO / 'outputs_postpull'

detector_summary = json.loads((OUT / 'support_gdino_full_lowthr' / 'support_eval_summary.json').read_text())
semantic_records = json.loads((OUT / 'semantic_query_only_support16' / 'semantic_pipeline_results.json').read_text())
support = json.loads((REPO / 'data' / 'vizwiz_object_localization' / 'support_set.json').read_text())
ann_by_image = {a['image_id']: a for a in support['annotations']}
cat_by_id = {c['id']: c['name'] for c in support['categories']}

def xywh_to_xyxy(box):
    x, y, w, h = box
    return [x, y, x + w, y + h]

def iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return 0.0 if union <= 0 else inter / union

rows = []
ious = []
hits = 0
nonempty = 0
for rec in semantic_records:
    ann = ann_by_image[rec['image_id']]
    gt_xyxy = xywh_to_xyxy(ann['bbox'])
    best = 0.0
    best_category = None
    for result in rec.get('results', []):
        val = iou(gt_xyxy, xywh_to_xyxy(result['bbox']))
        if val > best:
            best = val
            best_category = result.get('matched_category')
    if rec.get('results'):
        nonempty += 1
    if best >= 0.5:
        hits += 1
    ious.append(best)
    rows.append({
        'image_id': rec['image_id'],
        'gt_label': cat_by_id[ann['category_id']],
        'semantic_family': rec['semantic_family'],
        'selected': len(rec.get('results', [])),
        'best_iou': best,
        'best_category': best_category,
        'null_likely': rec['null_likely'],
    })

semantic_summary = {
    'num_images': len(rows),
    'mean_iou': sum(ious) / len(ious),
    'hit50': hits / len(rows),
    'num_nonempty_results': nonempty,
}

pd.DataFrame([
    {'run': 'detector_lowthr_support16', **detector_summary},
    {'run': 'semantic_query_only_support16', **semantic_summary},
])

,run,checkpoint_path,num_images,mean_iou,hit50,num_nonempty_results
0,detector_lowthr_support16,/home/david/semantic_vlm_privacy/checkpoints/g...,16,0.972406,1.0000,16
1,semantic_query_only_support16,NaN,16,0.670402,0.6875,11


In [2]:
semantic_df = pd.DataFrame(rows).sort_values(['best_iou', 'selected'], ascending=[True, True])
semantic_df

,image_id,gt_label,semantic_family,selected,best_iou,best_category,null_likely
0,1166,bank_statement,transactional paper,0,0.000000,NaN,False
3,730,condom_box,health indicator,0,0.000000,NaN,False
4,668,condom_with_plastic_bag,health indicator,0,0.000000,NaN,False
11,662,empty_pill_bottle,health indicator,0,0.000000,NaN,False
14,136,tattoo_sleeve,null,0,0.000000,NaN,True
5,210,credit_or_debit_card,printed financial card,2,0.957250,credit or debit card,False
15,130,transcript,transactional paper,2,0.957905,NaN,False
2,225,business_card,visual identity card,1,0.966439,business card,False
6,674,doctors_prescription,statement or report document,1,0.966665,NaN,False
12,545,pregnancy_test_box,health indicator,1,0.974223,pregnancy test box,False


## Failure-case prompt variants

The next table compares the controller's generated semantic family and detector cues for the main failures under:
- the current baseline query-only prompt
- a targeted prompt variant that pushes health-indicator objects toward concrete lead cues and explicitly tells the model to label patterned arm sleeves as `tattoo sleeve`

In [3]:
variant_rows = json.loads((OUT / 'controller_prompt_variants_failure_cases.json').read_text())
variant_df = pd.DataFrame(variant_rows)
variant_df[['variant', 'image_id', 'gt_label', 'family', 'proposal_prompts', 'null_likely', 'document_text_hint']]

,variant,image_id,gt_label,family,proposal_prompts,null_likely,document_text_hint
0,baseline_query_prompt,1166,bank statement,transactional paper,"[receipt, printed paper, blurry document, tran...",False,generic paper
1,baseline_query_prompt,730,condom box,health indicator,"[green box, contraceptive box, pregnancy test ...",False,
2,baseline_query_prompt,668,condom with plastic bag,health indicator,"[small packet, plastic wrapper, health indicat...",False,
3,baseline_query_prompt,662,empty pill bottle,health indicator,"[pill bottle, prescription bottle, blue pill b...",False,generic paper
4,baseline_query_prompt,136,tattoo sleeve,null,"[null, no private object identified]",True,
5,concrete_health_tattoo_prompt,1166,bank statement,transactional paper,"[receipt, transactional paper, blurry receipt ...",False,generic paper
6,concrete_health_tattoo_prompt,730,condom box,health indicator,"[pregnancy test box, green box with visible te...",False,
7,concrete_health_tattoo_prompt,668,condom with plastic bag,health indicator,"[health indicator, wrapped condom packet, smal...",False,
8,concrete_health_tattoo_prompt,662,empty pill bottle,health indicator,"[pill bottle, prescription bottle, blue pill b...",False,generic paper
9,concrete_health_tattoo_prompt,136,tattoo sleeve,tattoo sleeve,"[arm sleeve, tattoo sleeve, patterned sleeve, ...",False,


## Observations

- The query-only semantic pipeline is substantially weaker than the recall-first detector baseline on the support set.
- It succeeds on many document-like categories, but it over-prunes or misframes several health-indicator and tattoo cases.
- The prompt variant clearly fixed the `tattoo_sleeve` controller failure: it changed the family from `null` to `tattoo sleeve` and surfaced concrete sleeve cues.
- The same prompt variant did not fix the blurry `bank_statement` case, and it actually made the `condom_box` cue drift harder toward `pregnancy test box`.
- So the current bottleneck is split: some failures are cue-generation problems, but others are category ambiguity or over-strict later filtering.

## Long Queue

A longer unattended experiment queue can now be run on the restored environment. Suggested targets:
- query-only semantic on the full dev set
- support-conditioned semantic on the full dev set
- support-conditioned semantic without document-text enrichment
- support-conditioned semantic with hybrid category scores enabled

Use the GPU with the most free memory and keep `--disable-sam` for the first long runs.